
# Thí nghiệm 01 — Xây dựng kiến trúc mô hình dự báo Lorenz

## Mục tiêu

Notebook này định nghĩa các kiến trúc mạng nơ-ron được sử dụng trong thí nghiệm dự báo chuỗi thời gian Lorenz.

Các mô hình kết hợp ba thành phần chính:

1. **Bộ trích xuất đặc trưng:** sử dụng kiến trúc dựa trên Transformer để mã hóa chuỗi đầu vào thành biểu diễn tiềm ẩn.
2. **Toán tử Koopman:** biến đổi biểu diễn tiềm ẩn nhằm mô hình hóa sự tiến hóa của trạng thái.
3. **Bộ giải mã tuyến tính:** chuyển biểu diễn sau biến đổi thành các giá trị dự báo.

Thí nghiệm xây dựng ba biến thể sử dụng các bộ trích xuất đặc trưng khác nhau:

- PatchTST-based.
- Autoformer-based.
- Informer-based.

## Phạm vi

Notebook chỉ chứa định nghĩa các thành phần và lớp mô hình.

Quá trình huấn luyện, đánh giá kết quả, trực quan hóa dự báo và quản lý checkpoint được tách sang các phần riêng.

## Lưu ý về mức độ tái lập

Các backbone trong notebook là những phiên bản triển khai đơn giản hóa, dựa trên ý tưởng của PatchTST, Autoformer và Informer.

Chúng không mặc nhiên tương đương hoàn toàn với các kiến trúc chuẩn được công bố trong các bài báo gốc.

Các khác biệt quan trọng được ghi rõ tại từng thành phần để tránh diễn giải sai trong báo cáo.


In [ ]:

import torch
import torch.nn as nn

# Tạo ma trận trực giao Q bằng phép phân rã QR của PyTorch
def householder_orthogonalization(W):
    Q, _ = torch.linalg.qr(W)
    return Q



## 1. Toán tử Koopman ổn định nghiêm ngặt

### 1.1. Ý tưởng

Trong bài toán dự báo chuỗi thời gian phi tuyến, sự tiến hóa trực tiếp của các biến quan sát có thể rất phức tạp.

Cách tiếp cận dựa trên Koopman tìm cách biểu diễn hệ động lực trong một không gian tiềm ẩn (*latent space*), tại đó quá trình chuyển trạng thái được mô hình hóa bằng một phép biến đổi tuyến tính.

Gọi $\mathbf{z}_t$ là biểu diễn tiềm ẩn tại thời điểm $t$, ta xét:

$$
\mathbf{z}_{t+1}=K\mathbf{z}_t
$$

Trong đó $K$ là ma trận toán tử Koopman.

Mã nguồn sử dụng quy ước vector hàng trong PyTorch nên phép biến đổi được viết thành:

`z_next = z @ K.T`

### 1.2. Cấu trúc toán tử

Ma trận Koopman được xây dựng theo dạng:

$$
K=U\operatorname{diag}(s)V^\top
$$

Trong đó:

- $U$ và $V$ là các ma trận trực giao được tạo từ phép phân rã QR.
- $s$ là vector chứa các giá trị bị chặn.
- $\operatorname{diag}(s)$ là ma trận đường chéo tạo từ vector $s$.

Các giá trị trên đường chéo được xác định bởi:

$$
s_i=\rho_{\max}\operatorname{sigmoid}(a_i)
$$

Với $\rho_{\max}=0.99$, ta có:

$$
0<s_i<0.99
$$

### 1.3. Điều kiện ổn định

Do $U$ và $V$ trực giao, các giá trị kỳ dị của $K$ được khống chế bởi các giá trị $s_i$.

Vì vậy:

$$
\|K\|_2<0.99<1
$$

Suy ra phép biến đổi không làm tăng chuẩn Euclid của vector tiềm ẩn:

$$
\|K\mathbf{z}\|_2\leq 0.99\|\mathbf{z}\|_2
$$

Điều này tạo ra một ràng buộc co đối với phép biến đổi trong không gian tiềm ẩn.

### 1.4. Các thành phần có thể học

Lớp `StrictStableKoopmanOperator` chứa ba nhóm tham số:

- `U_raw`: ma trận tham số dùng để xây dựng $U$.
- `V_raw`: ma trận tham số dùng để xây dựng $V$.
- `S_unconstrained`: vector tham số được biến đổi bằng hàm sigmoid.

Các tham số này được tối ưu trong quá trình huấn luyện.

### 1.5. Đầu vào và đầu ra

**Đầu vào:** tensor biểu diễn tiềm ẩn có kích thước `(batch_size, latent_dim)`.

**Đầu ra:**

- `z_next`: biểu diễn sau phép biến đổi Koopman.
- `K`: ma trận Koopman được xây dựng trong lần truyền xuôi.
- `S`: vector các giá trị đường chéo đã được khống chế.

**Lưu ý:** Vector `S` không phải vector trị riêng của ma trận $K$. Trong cách xây dựng này, nó tương ứng với các giá trị kỳ dị của toán tử, sai khác chỉ có thể xuất hiện do sai số số học.


In [2]:

class StrictStableKoopmanOperator(nn.Module):
    def __init__(self, latent_dim, max_spectral_radius=0.99):
        super().__init__()

        self.latent_dim = latent_dim
        self.max_spectral_radius = max_spectral_radius

        self.U_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.V_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.S_unconstrained = nn.Parameter(
            torch.randn(latent_dim)
        )

    def forward(self, z):
        U = householder_orthogonalization(self.U_raw)
        V = householder_orthogonalization(self.V_raw)

        S = torch.sigmoid(self.S_unconstrained) * self.max_spectral_radius

        K = U @ torch.diag(S) @ V.T

        z_next = z @ K.T

        return z_next, K, S



## 2. Bộ trích xuất đặc trưng dựa trên PatchTST

### 2.1. Vai trò

Bộ trích xuất đặc trưng có nhiệm vụ biến đổi chuỗi thời gian đầu vào thành biểu diễn trong không gian tiềm ẩn.

Trong biến thể này, thành phần trích xuất đặc trưng được xây dựng bằng các tầng Transformer Encoder.

### 2.2. Các thành phần

Lớp `PatchTSTBackbone` gồm:

1. **Lớp nhúng tuyến tính (`embedding`):** ánh xạ vector đầu vào sang không gian có số chiều `d_model`.
2. **Tham số mã hóa vị trí (`pos_encoding`):** cộng một vector tham số có thể học vào các biểu diễn đầu vào.
3. **Transformer Encoder:** thực hiện biến đổi đặc trưng thông qua các tầng attention và mạng truyền thẳng.

### 2.3. Cấu hình trong thí nghiệm Lorenz

Khi được khởi tạo bên trong `PatchTSTWrapper`, backbone có cấu hình:

| Tham số | Giá trị |
|---|---|
| Số chiều đầu vào | 450 |
| Số chiều tiềm ẩn | 16 |
| Số attention head | 2 |
| Số chiều tầng truyền thẳng | 64 |
| Số tầng Transformer Encoder | 2 |

### 2.4. Đặc điểm triển khai

Trong mã nguồn hiện tại, `PatchTSTWrapper` trải phẳng toàn bộ cửa sổ gồm 150 bước thời gian và 3 biến thành một vector có 450 phần tử.

Vector này được đưa vào backbone dưới dạng **một token duy nhất**.

Do đó, đây là một phiên bản đơn giản hóa dựa trên Transformer, chưa thực hiện việc phân chia thành nhiều patch và khai thác attention giữa các patch như PatchTST chuẩn.

Ngoài ra, vector `pos_encoding` có kích thước `(1, d_model)` được cộng chung vào các token. Với cấu hình chỉ một token, thành phần này không biểu diễn được quan hệ vị trí giữa nhiều patch.

Những giới hạn này cần được phân biệt rõ khi so sánh với kiến trúc được mô tả trong tài liệu nghiên cứu.


In [ ]:
# Bộ trích xuất đặc trưng dựa trên PatchTST
class PatchTSTBackbone(nn.Module):
    def __init__(self, input_dim, d_model, num_heads, dim_feedforward, num_layers):
        super().__init__()

        self.embedding = nn.Linear(input_dim, d_model)

        self.pos_encoding = nn.Parameter(torch.randn(1, d_model)) # Vector nhúng có thể học, dùng chung cho các vị trí

        layers = nn.TransformerEncoderLayer(
            d_model, num_heads, dim_feedforward, batch_first=True
        )

        self.encoder = nn.TransformerEncoder(layers, num_layers)

    def forward(self, x):
        x = self.embedding(x) + self.pos_encoding
        return self.encoder(x)



## 3. Mô hình kết hợp PatchTST và toán tử Koopman

### 3.1. Kiến trúc tổng thể

Lớp `PatchTSTWrapper` kết hợp ba thành phần:

1. `PatchTSTBackbone`: trích xuất đặc trưng từ chuỗi đầu vào.
2. `StrictStableKoopmanOperator`: biến đổi biểu diễn trong không gian tiềm ẩn.
3. `Linear Decoder`: ánh xạ biểu diễn tiềm ẩn thành các giá trị dự báo.

Luồng xử lý:

**Chuỗi đầu vào → Backbone → Biểu diễn tiềm ẩn → Toán tử Koopman → Bộ giải mã → Dự báo**

### 3.2. Dữ liệu đầu vào

Mỗi mẫu Lorenz có kích thước:

`(150, 3)`

Trong bước truyền xuôi, dữ liệu được trải phẳng thành vector có 450 phần tử và thêm chiều token:

`(batch_size, 1, 450)`

### 3.3. Trích xuất biểu diễn tiềm ẩn

Backbone tạo biểu diễn có số chiều 16.

Sau Transformer Encoder, mô hình áp dụng phép lấy trung bình theo chiều token bằng `mean(dim=1)`.

Kết quả là vector tiềm ẩn:

`(batch_size, 16)`

### 3.4. Biến đổi Koopman

Vector tiềm ẩn được đưa qua toán tử Koopman để tạo biểu diễn mới, vẫn có 16 chiều.

Toán tử được ràng buộc về chuẩn phổ nhằm hạn chế sự khuếch đại trong không gian tiềm ẩn.

### 3.5. Giải mã dự báo

Bộ giải mã tuyến tính ánh xạ biểu diễn 16 chiều thành vector đầu ra có kích thước:

$$
5\times3=15
$$

Vector này chứa các giá trị dự báo cho 5 bước thời gian tương lai của ba biến Lorenz.

### 3.6. Kết quả trả về

Phương thức `forward()` trả về:

- `out`: vector dự báo.
- `latent`: biểu diễn tiềm ẩn trước Koopman.
- `latent_next`: biểu diễn tiềm ẩn sau Koopman.
- `K`: ma trận Koopman.
- `eigvals`: biến chứa các giá trị kỳ dị được khống chế, không phải trị riêng của ma trận Koopman.

Tên biến `eigvals` được giữ nguyên để bảo toàn giao diện với mã nguồn huấn luyện hiện có.


In [7]:

# DeepKoopFormer - PatchTST Wrapper
class PatchTSTWrapper(nn.Module):
    def __init__(self, patch_length, horizon):
        super().__init__()

        self.backbone = PatchTSTBackbone(
            patch_length * 3, 16, 2, 64, 2
        )

        self.koopman = StrictStableKoopmanOperator(16)

        self.fc = nn.Linear(16, horizon * 3)

    def forward(self, x):
        x = x.reshape(x.shape[0], 1, -1)

        latent = self.backbone(x).mean(dim=1)

        latent_next, K, eigvals = self.koopman(latent)

        out = self.fc(latent_next)

        return out, latent, latent_next, K, eigvals




## 4. Mã hóa vị trí cho các mô hình chuỗi thời gian

### 4.1. Mục đích

Transformer không tự biểu diễn trực tiếp thứ tự thời gian của các phần tử đầu vào. Vì vậy, một cách phổ biến để bổ sung thông tin thứ tự là sử dụng mã hóa vị trí (*positional encoding*).

Trong notebook, lớp `PositionalEncoding` xây dựng các vector vị trí từ hàm sin và cos với các tần số khác nhau.

### 4.2. Công thức

Với vị trí $pos$ và chiều biểu diễn $i$, mã hóa vị trí được xác định bởi:

$$
PE_{(pos,2i)} =
\sin\left(\frac{pos}{10000^{2i/d_{\mathrm{model}}}}\right)
$$

$$
PE_{(pos,2i+1)} =
\cos\left(\frac{pos}{10000^{2i/d_{\mathrm{model}}}}\right)
$$

Trong đó $d_{\mathrm{model}}$ là số chiều biểu diễn.

### 4.3. Cách triển khai

Lớp `PositionalEncoding`:

1. Khởi tạo bảng mã hóa vị trí có độ dài tối đa `max_len`.
2. Tính các giá trị sin và cos cho từng vị trí.
3. Lưu bảng dưới dạng buffer, không phải tham số có thể học.
4. Cộng các vector vị trí tương ứng vào biểu diễn đầu vào.

### 4.4. Phạm vi sử dụng

Lớp này được sử dụng trong các backbone dựa trên Autoformer và Informer trong notebook.

Riêng `PatchTSTBackbone` sử dụng một tham số `pos_encoding` có thể học, được định nghĩa độc lập và không gọi lớp `PositionalEncoding` này.


In [ ]:

import numpy as np

# Mã hóa thông tin vị trí bằng các hàm sin và cos
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-np.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)
    
    



## 5. Phân rã chuỗi thời gian

### 5.1. Mục đích

Phân rã chuỗi thời gian (*series decomposition*) là một thành phần được sử dụng trong backbone dựa trên Autoformer.

Ý tưởng là tách chuỗi đầu vào thành hai thành phần:

- **Xu hướng (trend):** biểu diễn biến thiên được làm trơn.
- **Phần dư (residual):** phần chênh lệch giữa chuỗi ban đầu và xu hướng.

Trong mã nguồn, phần dư được đặt tên là `seasonal`, nhưng không có bước kiểm tra riêng để khẳng định toàn bộ phần dư thực sự là thành phần mùa vụ.

### 5.2. Phương pháp

Thành phần xu hướng được ước lượng bằng phép trung bình trượt thông qua `AvgPool1d`.

$$
X_{\text{trend}}=\operatorname{MovingAverage}(X)
$$

Phần dư được xác định bởi:

$$
X_{\text{residual}}=X-X_{\text{trend}}
$$

Do đó:

$$
X=X_{\text{trend}}+X_{\text{residual}}
$$

### 5.3. Cấu hình triển khai

Lớp `SeriesDecomposition` sử dụng:

- `AvgPool1d` để tính trung bình trượt.
- `stride=1` để duy trì bước dịch một phần tử.
- `padding=kernel_size // 2` để xử lý biên chuỗi.

Khi được gọi trong `AutoformerBackbone`, kích thước cửa sổ trung bình trượt là 3.

PyTorch sử dụng giá trị đệm bằng 0 trong phép `AvgPool1d` với cấu hình hiện tại, vì vậy kết quả tại biên có thể chịu ảnh hưởng của cơ chế đệm.

### 5.4. Vai trò trong backbone

Hàm `forward()` trả về cả xu hướng và phần dư.

Tuy nhiên, backbone hiện tại chỉ đưa phần dư vào các tầng nhúng và Transformer Encoder.

Thành phần xu hướng chưa được sử dụng trong nhánh dự báo riêng như ở một số kiến trúc phân rã chuỗi thời gian đầy đủ.


In [ ]:

# Phân rã chuỗi thành thành phần xu hướng và phần dư.
class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size=25):
        super().__init__()

        self.moving_avg = nn.AvgPool1d(
            kernel_size, # Kích thước cửa sổ trung bình trượt
            stride=1, # Dịch cửa sổ một phần tử mỗi lần
            padding=kernel_size // 2 # Số phần tử đệm ở mỗi phía
        )

    def forward(self, x):
        trend = self.moving_avg(x.transpose(1, 2)).transpose(1, 2)

        return trend, x - trend


Original: tensor([2., 4., 9., 5., 1.])
Trend: tensor([2., 5., 6., 5., 2.])
Seasonal: tensor([ 0., -1.,  3.,  0., -1.])
Reconstructed: tensor([2., 4., 9., 5., 1.])



## 6. Bộ trích xuất đặc trưng dựa trên Autoformer

### 6.1. Ý tưởng

Autoformer là một hướng tiếp cận dự báo chuỗi thời gian có sử dụng cơ chế phân rã chuỗi để hỗ trợ mô hình hóa các thành phần có cấu trúc khác nhau.

Trong thí nghiệm này, `AutoformerBackbone` sử dụng một kiến trúc đơn giản hóa, kết hợp phân rã chuỗi thời gian với Transformer Encoder.

### 6.2. Các thành phần

Backbone gồm bốn bước:

1. **Phân rã chuỗi:** sử dụng `SeriesDecomposition` để tính xu hướng và phần dư.
2. **Nhúng tuyến tính:** chuyển mỗi giá trị đầu vào sang biểu diễn có số chiều 16.
3. **Mã hóa vị trí:** bổ sung thông tin vị trí cho các biểu diễn.
4. **Transformer Encoder:** xử lý chuỗi biểu diễn và tổng hợp thành vector tiềm ẩn.

### 6.3. Quy trình truyền xuôi

Đầu vào được phân rã:

`trend, seasonal = self.decomp(x)`

Trong đó `seasonal` thực chất là phần dư sau khi trừ xu hướng.

Chỉ phần dư được đưa qua lớp nhúng và mã hóa vị trí:

`seasonal = self.pos_encoding(self.embed(seasonal))`

Sau đó, Transformer Encoder tạo các biểu diễn theo chuỗi.

Cuối cùng, phép lấy trung bình theo chiều thời gian `mean(dim=1)` được sử dụng để thu được một vector tiềm ẩn cho mỗi mẫu.

### 6.4. Cấu hình

| Thành phần | Giá trị |
|---|---|
| Kích thước cửa sổ trung bình trượt | 3 |
| Số chiều biểu diễn | 16 |
| Số attention head | 2 |
| Số chiều tầng truyền thẳng | 32 |
| Số tầng Transformer Encoder | 2 |

### 6.5. Giới hạn triển khai

Backbone hiện tại không triển khai đầy đủ cơ chế Auto-Correlation của Autoformer gốc.

Ngoài ra, thành phần xu hướng được tính ra nhưng không đi vào một nhánh dự báo riêng.

Vì vậy, mô hình này được mô tả là **biến thể dựa trên Autoformer**, không phải bản tái lập đầy đủ của kiến trúc Autoformer chuẩn.


In [ ]:
# Bộ trích xuất đặc trưng dựa trên Autoformer
class AutoformerBackbone(nn.Module):
    def __init__(self, input_len, d_model):
        super().__init__()

        self.decomp = SeriesDecomposition(3)

        self.embed = nn.Linear(1, d_model)

        self.pos_encoding = PositionalEncoding(d_model)

        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model, 2, 32, batch_first=True
            ),
            2
        )

    def forward(self, x):
        trend, seasonal = self.decomp(x)

        seasonal = self.pos_encoding(
            self.embed(seasonal)
        )

        latent = self.encoder(seasonal).mean(dim=1)

        return latent



## 7. Mô hình kết hợp Autoformer và toán tử Koopman

### 7.1. Kiến trúc

Lớp `AutoformerWrapper` kết hợp:

- `AutoformerBackbone` để trích xuất đặc trưng.
- `StrictStableKoopmanOperator` để biến đổi biểu diễn tiềm ẩn.
- Một tầng tuyến tính để tạo giá trị dự báo.

### 7.2. Xử lý đầu vào

Đầu vào ban đầu có kích thước:

`(batch_size, 150, 3)`

Mô hình trải phẳng chuỗi rồi tổ chức thành một chuỗi một biến:

`(batch_size, 450, 1)`

Điều này có nghĩa là ba biến Lorenz được ghép nối theo thứ tự phần tử vào cùng một chuỗi trước khi đi qua backbone.

Đây là đặc điểm cụ thể của cách triển khai hiện tại, cần phân biệt với những mô hình xử lý ba kênh đầu vào riêng biệt.

### 7.3. Biểu diễn tiềm ẩn và dự báo

Backbone tạo vector tiềm ẩn có 16 chiều.

Vector này được đưa qua toán tử Koopman ổn định, sau đó qua bộ giải mã tuyến tính.

Với chân trời dự báo bằng 5 và số biến bằng 3, đầu ra là vector gồm 15 giá trị.

### 7.4. Các giá trị trả về

Mô hình trả về:

- Giá trị dự báo.
- Biểu diễn tiềm ẩn trước Koopman.
- Biểu diễn tiềm ẩn sau Koopman.
- Ma trận Koopman.
- Vector các giá trị kỳ dị được khống chế.

Các giá trị này được giữ lại để phục vụ huấn luyện và các phép đánh giá khi cần thiết.


In [ ]:
# Mô hình kết hợp backbone dựa trên Autoformer và toán tử Koopman
class AutoformerWrapper(nn.Module):
    def __init__(self, patch_length, horizon):
        super().__init__()

        self.backbone = AutoformerBackbone(
            patch_length * 3, 16
        )

        self.koopman = StrictStableKoopmanOperator(16)

        self.fc = nn.Linear(16, horizon * 3)

    def forward(self, x):
        x = x.reshape(x.shape[0], -1, 1)

        latent = self.backbone(x)

        latent_next, K, eigvals = self.koopman(latent)

        out = self.fc(latent_next)

        return out, latent, latent_next, K, eigvals



## 8. Bộ trích xuất đặc trưng dựa trên Informer

### 8.1. Mục đích

Lớp `InformerSparse` được sử dụng để trích xuất đặc trưng từ chuỗi thời gian đa biến bằng kiến trúc Transformer Encoder.

Đầu vào là một chuỗi các quan sát gồm ba biến Lorenz.

### 8.2. Các thành phần

Backbone gồm:

1. Lớp nhúng tuyến tính từ ba biến đầu vào sang không gian 16 chiều.
2. Lớp mã hóa vị trí bằng sin và cos.
3. Hai tầng Transformer Encoder.
4. Phép lấy trung bình theo chiều thời gian để tạo biểu diễn tiềm ẩn.

### 8.3. Dữ liệu đầu vào và đầu ra

Đầu vào:

`(batch_size, 150, 3)`

Đầu ra:

`(batch_size, 16)`

Khác với hai wrapper trước, biến thể này không trải phẳng toàn bộ chuỗi trước khi đưa vào backbone.

Mỗi thời điểm được biểu diễn thành một token gồm ba biến đầu vào.

### 8.4. Cấu hình

| Thành phần | Giá trị |
|---|---|
| Số biến đầu vào | 3 |
| Số chiều biểu diễn | 16 |
| Số attention head | 2 |
| Số chiều tầng truyền thẳng | 32 |
| Số tầng Transformer Encoder | 2 |

### 8.5. Giới hạn triển khai

Mặc dù lớp có tên `InformerSparse`, mã nguồn hiện tại sử dụng `nn.TransformerEncoder` tiêu chuẩn của PyTorch.

Không có cơ chế ProbSparse attention được cài đặt riêng trong lớp này.

Vì vậy, đây là một backbone Transformer đơn giản hóa, được sử dụng như biến thể đối chiếu dựa trên Informer.

Không nên khẳng định mô hình đã tái lập đầy đủ cơ chế attention thưa của Informer gốc.


In [ ]:
# Bộ trích xuất đặc trưng Transformer dựa trên Informer
class InformerSparse(nn.Module):
    def __init__(self, input_dim, d_model):
        super().__init__()

        self.embed = nn.Linear(input_dim, d_model)

        self.pos_encoding = PositionalEncoding(d_model)

        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model, 2, 32, batch_first=True
            ),
            2
        )

    def forward(self, x):
        x = self.pos_encoding(self.embed(x))

        return self.encoder(x).mean(dim=1)


Input shape: torch.Size([2, 150, 3])
Latent shape: torch.Size([2, 16])



## 9. Mô hình kết hợp Informer và toán tử Koopman

### 9.1. Kiến trúc

Lớp `InformerWrapper` xây dựng mô hình dự báo bằng cách kết hợp:

- Bộ trích xuất đặc trưng `InformerSparse`.
- Toán tử `StrictStableKoopmanOperator`.
- Bộ giải mã tuyến tính.

### 9.2. Quy trình dự báo

Dữ liệu đầu vào có dạng:

`(batch_size, 150, 3)`

Đầu tiên, backbone biến chuỗi đầu vào thành vector tiềm ẩn 16 chiều.

Tiếp theo, toán tử Koopman thực hiện phép biến đổi tuyến tính có ràng buộc ổn định trong không gian tiềm ẩn.

Cuối cùng, tầng tuyến tính giải mã thành vector dự báo gồm 15 phần tử, tương ứng với 5 bước tương lai của ba biến trạng thái.

### 9.3. Đặc điểm so với các biến thể khác

- Biến thể PatchTST trải phẳng cửa sổ thành một token duy nhất.
- Biến thể Autoformer trải phẳng dữ liệu thành chuỗi một biến trước khi phân rã.
- Biến thể Informer giữ cấu trúc chuỗi gồm 150 thời điểm và 3 biến khi đưa vào backbone.

Ba biến thể cùng sử dụng toán tử Koopman ổn định nhưng khác nhau ở cách biểu diễn và trích xuất đặc trưng từ dữ liệu đầu vào.

### 9.4. Lưu ý khi đánh giá

Việc so sánh các mô hình phải dựa trên kết quả thực nghiệm đã lưu.

Chỉ từ cấu trúc kiến trúc, chưa thể kết luận biến thể nào có chất lượng dự báo tốt hơn.


In [ ]:
# Mô hình kết hợp backbone dựa trên Informer và toán tử Koopman
class InformerWrapper(nn.Module):
    def __init__(self, patch_length, horizon):
        super().__init__()

        self.backbone = InformerSparse(3, 16)

        self.koopman = StrictStableKoopmanOperator(16)

        self.fc = nn.Linear(16, horizon * 3)

    def forward(self, x):
        latent = self.backbone(x)

        latent_next, K, eigvals = self.koopman(latent)

        out = self.fc(latent_next)

        return out, latent, latent_next, K, eigvals
